# Ube Craze Sentiment Visualizations

This notebook generates visualizations from the final sentiment-scored dataset to illustrate the dynamics of Filipino gastronationalism (positive pride) versus potential cultural dilution (negative/neutral/exoticizing themes).

## 1. Imports and Config

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from nltk.util import ngrams
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from ube_craze_nlp.nlp.clean import CLUSTERING_STOPWORDS, remove_stopwords, tokenize_text
from ube_craze_nlp.utils.paths import FINAL_DATA_DIR, OUTPUTS_DIR, OUTPUTS_PLOTS_DIR, OUTPUTS_CLUSTERS_DIR, OUTPUTS_DOCS_DIR, ensure_dirs

ensure_dirs()

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

print(f"Figures will be exported to: {OUTPUTS_DIR}")

## 2. Load Scored Dataset

In [ ]:
final_file = FINAL_DATA_DIR / "sentiment_scored_comments.csv"

if not final_file.exists():
    print("❌ Scored dataset not found! Please run Notebook 03 first.")
    df_comments = pd.DataFrame()
else:
    df_comments = pd.read_csv(
        final_file, dtype={"comment_id": str, "reply_to_comment_id": str, "video_id": str}
    )
    df_comments["reply_to_comment_id"] = df_comments["reply_to_comment_id"].fillna("").astype(str)
    print(f"Loaded {len(df_comments)} scored comments.")
    display(df_comments.head())

## 3. Visualize Sentiment Distribution

We plot the overall sentiment distribution (Positive, Neutral, Negative) to understand public reception of the Ube craze on TikTok.

In [ ]:
if not df_comments.empty:
    # Compute counts and percentages
    sentiment_counts = df_comments["sentiment"].value_counts()
    sentiment_pct = df_comments["sentiment"].value_counts(normalize=True) * 100

    # Plot
    palette = {
        "positive": "#6a0dad",
        "neutral": "#a295a7",
        "negative": "#d46a83",
    }  # Purple, Grey, Muted Pink
    ax = sns.barplot(
        x=sentiment_counts.index,
        y=sentiment_counts.values,
        hue=sentiment_counts.index,
        palette=palette,
        legend=False,
    )

    plt.title(
        "TikTok Comment Sentiment on the Global Ube Craze", fontsize=14, fontweight="bold", pad=15
    )
    plt.xlabel("Sentiment Label", fontsize=12)
    plt.ylabel("Number of Comments", fontsize=12)

    # Add text labels on top of the bars
    for i, p in enumerate(ax.patches):
        height = p.get_height()
        percentage = sentiment_pct.iloc[i]
        ax.annotate(
            f"{int(height)} ({percentage:.1f}%)",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="center",
            xytext=(0, 9),
            textcoords="offset points",
            fontsize=11,
            fontweight="semibold",
        )

    sns.despine()
    plt.tight_layout()

    # Save plot
    fig_path = OUTPUTS_PLOTS_DIR / "sentiment_distribution.png"
    plt.savefig(fig_path, dpi=300)
    print(f"✅ Exported plot to: {fig_path}")
    plt.show()

## 4. Word Frequency Distributions (Pride vs. Context vs. Friction)

We extract high-frequency terms from positive comments (potential indicators of national pride and Gastronationalism), neutral comments (potential indicators of description, classification, or product context), and negative comments (potential indicators of commercialization, exoticization, cultural dilution, or friction).

In [ ]:
def get_frequent_words(texts_series, top_n=15):
    all_tokens = []
    for text in texts_series:
        tokens = tokenize_text(str(text))
        cleaned_tokens = remove_stopwords(tokens)
        all_tokens.extend(cleaned_tokens)
    return Counter(all_tokens).most_common(top_n)


if not df_comments.empty:
    # Separate positive, neutral, and negative comment texts
    pos_comments = df_comments[df_comments["sentiment"] == "positive"]["normalized_text"]
    neu_comments = df_comments[df_comments["sentiment"] == "neutral"]["normalized_text"]
    neg_comments = df_comments[df_comments["sentiment"] == "negative"]["normalized_text"]

    # Compute word counts
    pos_words = get_frequent_words(pos_comments)
    neu_words = get_frequent_words(neu_comments)
    neg_words = get_frequent_words(neg_comments)

    # --- Plot Positive Words ---
    df_pos_words = pd.DataFrame(pos_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=df_pos_words, x="count", y="word", hue="word", palette="Purples_r", legend=False
    )
    plt.title(
        "Top Words in Positive Comments (Cultural Pride / Gastronationalism)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    pos_fig_path = OUTPUTS_PLOTS_DIR / "word_frequency_pride.png"
    plt.savefig(pos_fig_path, dpi=300)
    print(f"✅ Exported positive words plot to: {pos_fig_path}")
    plt.show()

    # --- Plot Neutral Words ---
    df_neu_words = pd.DataFrame(neu_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_neu_words, x="count", y="word", hue="word", palette="Blues_r", legend=False)
    plt.title(
        "Top Words in Neutral Comments (Context / Description / Products)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    neu_fig_path = OUTPUTS_PLOTS_DIR / "word_frequency_neutral.png"
    plt.savefig(neu_fig_path, dpi=300)
    print(f"✅ Exported neutral words plot to: {neu_fig_path}")
    plt.show()

    # --- Plot Negative Words ---
    df_neg_words = pd.DataFrame(neg_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_neg_words, x="count", y="word", hue="word", palette="Reds_r", legend=False)
    plt.title(
        "Top Words in Negative Comments (Cultural Friction / Resistance)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    neg_fig_path = OUTPUTS_PLOTS_DIR / "word_frequency_exotic.png"
    plt.savefig(neg_fig_path, dpi=300)
    print(f"✅ Exported negative words plot to: {neg_fig_path}")
    plt.show()

## 5. N-Gram (Bigram & Trigram) Analysis

We extract and visualize multi-word phrases (bigrams and trigrams) to capture meaningful combinations of words (e.g., "Trader Joes", "purple yam") that single-word unigram analysis breaks apart.

In [ ]:
def get_frequent_ngrams(texts_series, n, top_n=15):
    all_ngrams = []
    for text in texts_series:
        tokens = tokenize_text(str(text))
        cleaned_tokens = remove_stopwords(tokens)
        ngram_list = list(ngrams(cleaned_tokens, n))
        ngram_strings = [" ".join(gram) for gram in ngram_list]
        all_ngrams.extend(ngram_strings)
    return Counter(all_ngrams).most_common(top_n)


if not df_comments.empty:
    pos_comments = df_comments[df_comments["sentiment"] == "positive"]["normalized_text"]
    neu_comments = df_comments[df_comments["sentiment"] == "neutral"]["normalized_text"]
    neg_comments = df_comments[df_comments["sentiment"] == "negative"]["normalized_text"]

    # --- Bigrams ---
    pos_bigrams = get_frequent_ngrams(pos_comments, n=2)
    neu_bigrams = get_frequent_ngrams(neu_comments, n=2)
    neg_bigrams = get_frequent_ngrams(neg_comments, n=2)

    df_pos_bi = pd.DataFrame(pos_bigrams, columns=["phrase", "count"])
    df_neu_bi = pd.DataFrame(neu_bigrams, columns=["phrase", "count"])
    df_neg_bi = pd.DataFrame(neg_bigrams, columns=["phrase", "count"])

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))

    # Positive Bigrams
    sns.barplot(
        data=df_pos_bi,
        x="count",
        y="phrase",
        ax=axes[0],
        hue="phrase",
        palette="Purples_r",
        legend=False,
    )
    axes[0].set_title("Top Bigrams in Positive Comments (Pride)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Frequency")
    axes[0].set_ylabel("")

    # Neutral Bigrams
    sns.barplot(
        data=df_neu_bi,
        x="count",
        y="phrase",
        ax=axes[1],
        hue="phrase",
        palette="Blues_r",
        legend=False,
    )
    axes[1].set_title("Top Bigrams in Neutral Comments (Context)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Frequency")
    axes[1].set_ylabel("")

    # Negative Bigrams
    sns.barplot(
        data=df_neg_bi,
        x="count",
        y="phrase",
        ax=axes[2],
        hue="phrase",
        palette="Reds_r",
        legend=False,
    )
    axes[2].set_title("Top Bigrams in Negative Comments (Friction)", fontsize=12, fontweight="bold")
    axes[2].set_xlabel("Frequency")
    axes[2].set_ylabel("")

    plt.tight_layout()
    bi_fig_path = OUTPUTS_PLOTS_DIR / "word_frequency_bigrams.png"
    plt.savefig(bi_fig_path, dpi=300)
    print(f"✅ Exported 3-panel bigram plot to: {bi_fig_path}")
    plt.show()

    # --- Trigrams ---
    pos_trigrams = get_frequent_ngrams(pos_comments, n=3)
    neu_trigrams = get_frequent_ngrams(neu_comments, n=3)
    neg_trigrams = get_frequent_ngrams(neg_comments, n=3)

    df_pos_tri = pd.DataFrame(pos_trigrams, columns=["phrase", "count"])
    df_neu_tri = pd.DataFrame(neu_trigrams, columns=["phrase", "count"])
    df_neg_tri = pd.DataFrame(neg_trigrams, columns=["phrase", "count"])

    fig, axes = plt.subplots(1, 3, figsize=(26, 7))

    # Positive Trigrams
    sns.barplot(
        data=df_pos_tri,
        x="count",
        y="phrase",
        ax=axes[0],
        hue="phrase",
        palette="Purples_r",
        legend=False,
    )
    axes[0].set_title("Top Trigrams in Positive Comments (Pride)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Frequency")
    axes[0].set_ylabel("")

    # Neutral Trigrams
    sns.barplot(
        data=df_neu_tri,
        x="count",
        y="phrase",
        ax=axes[1],
        hue="phrase",
        palette="Blues_r",
        legend=False,
    )
    axes[1].set_title("Top Trigrams in Neutral Comments (Context)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Frequency")
    axes[1].set_ylabel("")

    # Negative Trigrams
    sns.barplot(
        data=df_neg_tri,
        x="count",
        y="phrase",
        ax=axes[2],
        hue="phrase",
        palette="Reds_r",
        legend=False,
    )
    axes[2].set_title(
        "Top Trigrams in Negative Comments (Friction)", fontsize=12, fontweight="bold"
    )
    axes[2].set_xlabel("Frequency")
    axes[2].set_ylabel("")

    plt.tight_layout()
    tri_fig_path = OUTPUTS_PLOTS_DIR / "word_frequency_trigrams.png"
    plt.savefig(tri_fig_path, dpi=300)
    print(f"✅ Exported 3-panel trigram plot to: {tri_fig_path}")
    plt.show()

## 6. Unsupervised Topic Clustering (K-Means)

We use TF-IDF vectorization and K-Means clustering to discover latent topics in the comments dataset automatically, then evaluate the sentiment breakdown across these different topics.

In [ ]:
if not df_comments.empty:
    # Filter to target language comments for text clustering
    df_target = df_comments[df_comments["detected_language"].isin(["en", "tl", None])].copy()
    texts = df_target["normalized_text"].fillna("").tolist()

    def get_clean_string(text):
        tokens = tokenize_text(str(text))
        cleaned = [t for t in remove_stopwords(tokens) if t not in CLUSTERING_STOPWORDS]
        return " ".join(cleaned)

    clean_texts = [get_clean_string(t) for t in texts]
    non_empty_indices = [i for i, t in enumerate(clean_texts) if t.strip() != ""]

    if len(non_empty_indices) < 6:
        print("⚠️ Too few comments with valid tokens to cluster. Skipping clustering.")
    else:
        df_cluster = df_target.iloc[non_empty_indices].copy().reset_index(drop=True)
        texts_for_tfidf = [clean_texts[i] for i in non_empty_indices]

        vectorizer = TfidfVectorizer(max_features=800, min_df=3, max_df=0.7)
        tfidf_matrix = vectorizer.fit_transform(texts_for_tfidf)

        # Apply TruncatedSVD (LSA)
        svd = TruncatedSVD(n_components=25, random_state=42)
        lsa_matrix = svd.fit_transform(tfidf_matrix)

        # --- Elbow Method calculation and plotting ---
        inertias = []
        k_range = range(2, min(13, len(df_cluster)))
        for k in k_range:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            km.fit(lsa_matrix)
            inertias.append(km.inertia_)

        plt.figure(figsize=(8, 5))
        plt.plot(k_range, inertias, marker="o", color="#6a0dad")
        plt.title(
            "K-Means Elbow Curve for Topic Clustering", fontsize=13, fontweight="bold", pad=15
        )
        plt.xlabel("Number of Clusters (K)", fontsize=11)
        plt.ylabel("Inertia (Sum of Squared Distances)", fontsize=11)
        plt.tight_layout()
        elbow_path = OUTPUTS_CLUSTERS_DIR / "kmeans_elbow_curve.png"
        plt.savefig(elbow_path, dpi=300)
        print(f"✅ Exported Elbow curve to: {elbow_path}")
        plt.show()
        plt.close()

        # --- Fit KMeans with K=7 ---
        n_clusters = min(7, len(df_cluster))
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_cluster["topic_cluster"] = kmeans.fit_predict(lsa_matrix)

        print("Top terms per cluster:")
        vocab = vectorizer.get_feature_names_out()
        import numpy as np

        cluster_labels = {}
        for i in range(n_clusters):
            mask = (df_cluster["topic_cluster"] == i).values
            cluster_tfidf = tfidf_matrix[mask]
            
            if cluster_tfidf.shape[0] > 0:
                mean_tfidf = np.asarray(cluster_tfidf.mean(axis=0)).flatten()
                sorted_indices = mean_tfidf.argsort()[::-1]
                top_terms = [vocab[idx] for idx in sorted_indices[:8]]
            else:
                top_terms = ["empty", "empty"]
            
            terms_str = ", ".join(top_terms)
            print(f"  Cluster {i}: {terms_str}")
            cluster_labels[i] = f"Topic {i} ({top_terms[0]}, {top_terms[1]})"

        df_cluster["topic_label"] = df_cluster["topic_cluster"].map(cluster_labels)

        # --- New Visualization: Cluster Sizes Distribution ---
        plt.figure(figsize=(10, 5))
        sizes = df_cluster["topic_cluster"].value_counts().sort_values(ascending=False)
        cluster_names = [cluster_labels[idx] for idx in sizes.index]
        sns.barplot(x=sizes.values, y=cluster_names, hue=cluster_names, palette="Purples_r", legend=False)
        plt.title("Distribution of Comments Across Topic Clusters", fontsize=13, fontweight="bold", pad=15)
        plt.xlabel("Number of Comments", fontsize=11)
        plt.ylabel("Topic Cluster", fontsize=11)
        plt.tight_layout()
        size_dist_path = OUTPUTS_CLUSTERS_DIR / "cluster_size_distribution.png"
        plt.savefig(size_dist_path, dpi=300)
        print(f"✅ Exported cluster size distribution plot to: {size_dist_path}")
        plt.show()
        plt.close()

        # --- New Visualization: Sentiment Distribution per Cluster (Stacked Bar) ---
        sent_per_cluster = df_cluster.groupby("topic_cluster")["sentiment"].value_counts(normalize=True).unstack(fill_value=0) * 100
        sent_per_cluster = sent_per_cluster.reindex(columns=["positive", "neutral", "negative"])
        sent_per_cluster.index = [cluster_labels[i] for i in sent_per_cluster.index]
        sent_per_cluster = sent_per_cluster.sort_index()

        colors_sent = ["#6a0dad", "#a295a7", "#d46a83"]
        sent_per_cluster.plot(kind="barh", stacked=True, color=colors_sent, figsize=(12, 6))
        plt.title("Sentiment Breakdown across Topic Clusters (%)", fontsize=13, fontweight="bold", pad=15)
        plt.xlabel("Percentage of Comments (%)", fontsize=11)
        plt.ylabel("Topic Cluster", fontsize=11)
        plt.legend(title="Sentiment", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        sent_stacked_path = OUTPUTS_CLUSTERS_DIR / "cluster_sentiment_stacked.png"
        plt.savefig(sent_stacked_path, dpi=300)
        print(f"✅ Exported cluster sentiment stacked plot to: {sent_stacked_path}")
        plt.show()
        plt.close()

        # --- New Visualization: Language Distribution per Cluster (Stacked Bar) ---
        lang_per_cluster = df_cluster.groupby("topic_cluster")["detected_language"].value_counts(normalize=True).unstack(fill_value=0) * 100
        lang_per_cluster.index = [cluster_labels[i] for i in lang_per_cluster.index]
        lang_per_cluster = lang_per_cluster.sort_index()

        colors_lang = ["#4A90E2", "#50E3C2", "#9B9B9B"]
        lang_per_cluster.plot(kind="barh", stacked=True, color=colors_lang[:len(lang_per_cluster.columns)], figsize=(12, 6))
        plt.title("Language Distribution across Topic Clusters (%)", fontsize=13, fontweight="bold", pad=15)
        plt.xlabel("Percentage of Comments (%)", fontsize=11)
        plt.ylabel("Topic Cluster", fontsize=11)
        plt.legend(title="Detected Language", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        lang_stacked_path = OUTPUTS_CLUSTERS_DIR / "cluster_language_stacked.png"
        plt.savefig(lang_stacked_path, dpi=300)
        print(f"✅ Exported cluster language stacked plot to: {lang_stacked_path}")
        plt.show()
        plt.close()

        # --- Multi-panel Word Cloud + Sentiment Grid Visualization ---
        import matplotlib.gridspec as gridspec
        from wordcloud import WordCloud

        fig = plt.figure(figsize=(16, 4 * n_clusters))
        gs = gridspec.GridSpec(n_clusters, 2, figure=fig, width_ratios=[1.2, 0.8])

        for i in range(n_clusters):
            # Filter texts for this cluster
            cluster_texts = df_cluster[df_cluster["topic_cluster"] == i]["normalized_text"].tolist()
            cluster_words = []
            for t in cluster_texts:
                tokens = tokenize_text(t)
                cleaned = [w for w in remove_stopwords(tokens) if w not in CLUSTERING_STOPWORDS]
                cluster_words.extend(cleaned)
            cluster_words_str = " ".join(cluster_words)

            # 1. Word Cloud Subplot & Save Individual Cloud
            if cluster_words_str.strip():
                wc = WordCloud(
                    background_color="white",
                    colormap="Purples",
                    width=600,
                    height=300,
                    max_words=50,
                    random_state=42,
                ).generate(cluster_words_str)
                wc.to_file(OUTPUTS_CLUSTERS_DIR / f"cluster_{i}_wordcloud.png")
                
                ax_wc = fig.add_subplot(gs[i, 0])
                ax_wc.imshow(wc, interpolation="bilinear")
            else:
                ax_wc = fig.add_subplot(gs[i, 0])
                ax_wc.text(0.5, 0.5, "No tokens available", ha="center", va="center")
            ax_wc.axis("off")
            ax_wc.set_title(
                f"Cluster {i} Word Cloud: {cluster_labels[i]}", fontsize=12, fontweight="bold"
            )

            # 2. Sentiment Subplot
            ax_sent = fig.add_subplot(gs[i, 1])
            cluster_sentiment = df_cluster[df_cluster["topic_cluster"] == i][
                "sentiment"
            ].value_counts()
            cluster_sentiment = cluster_sentiment.reindex(
                ["positive", "neutral", "negative"], fill_value=0
            )

            palette = {"positive": "#6a0dad", "neutral": "#a295a7", "negative": "#d46a83"}
            sns.barplot(
                x=cluster_sentiment.values,
                y=cluster_sentiment.index,
                ax=ax_sent,
                hue=cluster_sentiment.index,
                palette=palette,
                legend=False,
            )
            ax_sent.set_title(f"Cluster {i} Sentiment Distribution", fontsize=12, fontweight="bold")
            ax_sent.set_xlabel("Number of Comments")
            sns.despine(ax=ax_sent)

        plt.tight_layout()
        grid_fig_path = OUTPUTS_CLUSTERS_DIR / "topic_clusters_wordclouds_sentiment.png"
        plt.savefig(grid_fig_path, dpi=300)
        print(f"✅ Exported topic cluster grid plot to: {grid_fig_path}")
        plt.show()
        plt.close()

        # --- Export 50 Representative Samples (25 Top Liked + 25 Random) ---
        samples_path = OUTPUTS_DOCS_DIR / "cluster_samples.md"
        with open(samples_path, "w", encoding="utf-8") as sf:
            sf.write("# K-Means Cluster Representative Comment Samples\n\n")
            sf.write(
                "This document contains a hybrid of the top 25 most liked comments (viral resonance) "
                "and 25 random comments (general representation) for each of the 7 K-Means topic clusters.\n\n"
            )

            for i in range(n_clusters):
                sf.write(f"## Cluster {i} - {cluster_labels[i]}\n\n")
                cluster_df = df_cluster[df_cluster["topic_cluster"] == i]

                if not cluster_df.empty:
                    # 1. Top 25 Liked
                    sf.write("### Top 25 Liked Comments\n\n")
                    top_liked = cluster_df.nlargest(25, "likes")
                    for idx, row in enumerate(top_liked.itertuples(), 1):
                        text_clean = str(row.text).replace("\n", " ").strip()
                        sf.write(
                            f"{idx}. **@{row.username}** "
                            f"(Sentiment: `{row.sentiment}`, Likes: {row.likes}):  \n"
                        )
                        sf.write(f'   > "{text_clean}"\n\n')

                    # 2. 25 Random Comments (excluding the top liked ones)
                    sf.write("### 25 Random Comments (General Discourse)\n\n")
                    remaining_df = cluster_df.drop(index=top_liked.index)
                    if not remaining_df.empty:
                        sample_size = min(25, len(remaining_df))
                        random_samples = remaining_df.sample(n=sample_size, random_state=42)
                        for idx, row in enumerate(random_samples.itertuples(), 1):
                            text_clean = str(row.text).replace("\n", " ").strip()
                            sf.write(
                                f"{idx}. **@{row.username}** "
                                f"(Sentiment: `{row.sentiment}`, Likes: {row.likes}):  \n"
                            )
                            sf.write(f'   > "{text_clean}"\n\n')
                    else:
                        sf.write("*No additional comments to sample.*\n\n")
                else:
                    sf.write("*No comments in this cluster.*\n\n")
                sf.write("---\n\n")
        print(f"✅ Exported cluster sample comments to: {samples_path}")


## 7. Reply-Thread Sentiment Interaction

We construct parent-reply sentiment pairs to build an interaction heatmap, helping us study the conversational dynamics of digital gastronationalism (e.g., how the community responds to negative/exoticizing comments versus positive expressions of pride).

In [ ]:
if not df_comments.empty:
    df_replies = df_comments[df_comments["is_reply"]].copy()
    df_parents = df_comments[~df_comments["is_reply"]].copy()

    if df_replies.empty:
        print("⚠️ No replies found in the dataset. Skipping reply-thread sentiment interaction.")
    else:
        # Strip decimal parts if float representations somehow occurred
        df_replies["reply_to_comment_id"] = (
            df_replies["reply_to_comment_id"]
            .fillna("")
            .astype(str)
            .str.split(".")
            .str[0]
            .str.strip()
        )
        df_parents["comment_id"] = (
            df_parents["comment_id"].fillna("").astype(str).str.split(".").str[0].str.strip()
        )

        df_pairs = df_replies.merge(
            df_parents,
            left_on="reply_to_comment_id",
            right_on="comment_id",
            suffixes=("_reply", "_parent"),
        )

        if df_pairs.empty:
            print("⚠️ No matching parent comments found for replies. Skipping interaction matrix.")
        else:
            print(f"Constructed {len(df_pairs)} parent-reply sentiment pairs.")

            categories = ["positive", "neutral", "negative"]
            transition_matrix = pd.crosstab(
                df_pairs["sentiment_parent"], df_pairs["sentiment_reply"], dropna=False
            )

            transition_matrix = transition_matrix.reindex(
                index=categories, columns=categories, fill_value=0
            )
            transition_matrix_pct = (
                transition_matrix.div(transition_matrix.sum(axis=1), axis=0).fillna(0) * 100
            )

            plt.figure(figsize=(8, 6))
            sns.heatmap(
                transition_matrix_pct,
                annot=transition_matrix.astype(str)
                + "\n("
                + transition_matrix_pct.round(1).astype(str)
                + "%)",
                fmt="",
                cmap="Purples",
                cbar_kws={"label": "Percentage of replies (%)"},
                annot_kws={"size": 11, "weight": "bold"},
            )

            plt.title(
                "TikTok Comment Sentiment Interaction Matrix\n"
                "(Parent Sentiment vs. Reply Sentiment)",
                fontsize=13,
                fontweight="bold",
                pad=15,
            )
            plt.xlabel("Direct Reply Sentiment", fontsize=11)
            plt.ylabel("Parent Comment Sentiment", fontsize=11)
            plt.tight_layout()

            heatmap_path = OUTPUTS_PLOTS_DIR / "reply_sentiment_heatmap.png"
            plt.savefig(heatmap_path, dpi=300)
            print(f"✅ Exported reply sentiment heatmap to: {heatmap_path}")
            plt.show()